# Session 14 — Automated MLOps Pipeline for Retraining and Deploying Models using CI/CD with GCP Tools

**Goal:** close the loop the previous two sessions left open — trigger the Vertex AI
pipeline from Session 12 **automatically**, on a schedule or in response to new data,
using Cloud Build and Cloud Scheduler, instead of a human submitting it by hand.

## The full automation loop

1. **Cloud Scheduler** fires on a cron schedule (e.g. nightly).
2. It triggers a **Cloud Function** that checks whether retraining conditions are met
   (new data landed, or drift detected — Session 5/17's job).
3. If so, it submits the **Vertex AI Pipeline** from Session 12.
4. **Cloud Build**, triggered by a Git push (via a GitHub trigger, the GCP analogue of
   the GitHub Actions workflow in Session 10), rebuilds and redeploys the serving
   container whenever the pipeline code itself changes.

## Prerequisites

Needs a **GCP project** with Cloud Build, Cloud Scheduler, Cloud Functions, and
Vertex AI enabled — not available in this sandbox. Complete, correct reference code
below.

```bash
pip install google-cloud-aiplatform google-cloud-scheduler functions-framework
```

## Step 1 — A Cloud Function that decides whether to retrain

This is the "brain" of the automation: it checks a condition (here, new rows in a
BigQuery table since the last training run) and only submits the expensive pipeline
job if retraining is actually warranted.

In [ ]:
retrain_trigger_fn = '''\
import functions_framework
from google.cloud import aiplatform, bigquery

PROJECT_ID = "your-gcp-project-id"
REGION = "us-central1"
PIPELINE_TEMPLATE = "gs://your-bucket/heart_disease_pipeline.json"
MIN_NEW_ROWS = 500

@functions_framework.http
def check_and_retrain(request):
    bq = bigquery.Client(project=PROJECT_ID)
    query = \'\'\'
        SELECT COUNT(*) AS n_new_rows
        FROM `your_dataset.heart_disease_raw`
        WHERE ingestion_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY)
    \'\'\'
    result = list(bq.query(query).result())[0]

    if result.n_new_rows < MIN_NEW_ROWS:
        return f"Only {result.n_new_rows} new rows, below threshold; skipping retrain.", 200

    aiplatform.init(project=PROJECT_ID, location=REGION)
    job = aiplatform.PipelineJob(
        display_name="scheduled-retrain",
        template_path=PIPELINE_TEMPLATE,
        parameter_values={"raw_data_path": "gs://your-bucket/heart_disease_raw.csv"},
    )
    job.submit()
    return f"Retraining triggered: {job.resource_name} ({result.n_new_rows} new rows)", 200
'''
with open("session14_retrain_trigger.py", "w") as f:
    f.write(retrain_trigger_fn)
print(retrain_trigger_fn)

## Step 2 — Deploy the function and schedule it

`gcloud functions deploy` publishes the function as an HTTP endpoint;
`gcloud scheduler jobs create` calls that endpoint on a cron schedule — no server to
manage for either piece.

In [ ]:
deploy_commands = '''\
gcloud functions deploy check_and_retrain \\
  --runtime python311 \\
  --trigger-http \\
  --entry-point check_and_retrain \\
  --region us-central1 \\
  --no-allow-unauthenticated

gcloud scheduler jobs create http nightly-retrain-check \\
  --schedule="0 2 * * *" \\
  --uri="https://us-central1-YOUR_PROJECT.cloudfunctions.net/check_and_retrain" \\
  --http-method=POST \\
  --oidc-service-account-email=scheduler-invoker@YOUR_PROJECT.iam.gserviceaccount.com
'''
print(deploy_commands)
print("'0 2 * * *' = every day at 02:00 -- runs while traffic (and cost) is low.")

## Step 3 — Cloud Build for the serving container (the CI/CD half)

This is the GCP-native equivalent of the GitHub Actions workflow in Session 10: a
`cloudbuild.yaml` that builds and pushes a container image, triggered automatically
by a push to the repo.

In [ ]:
cloudbuild_yaml = '''\
steps:
  - name: "python:3.11"
    entrypoint: pip
    args: ["install", "-r", "requirements.txt"]

  - name: "python:3.11"
    entrypoint: pytest
    args: ["test_api.py", "-v"]

  - name: "gcr.io/cloud-builders/docker"
    args: ["build", "-t", "us-central1-docker.pkg.dev/$PROJECT_ID/ml-images/api:$COMMIT_SHA", "."]

  - name: "gcr.io/cloud-builders/docker"
    args: ["push", "us-central1-docker.pkg.dev/$PROJECT_ID/ml-images/api:$COMMIT_SHA"]

  - name: "gcr.io/google.com/cloudsdktool/cloud-sdk"
    entrypoint: gcloud
    args:
      - run
      - deploy
      - heart-disease-api
      - --image=us-central1-docker.pkg.dev/$PROJECT_ID/ml-images/api:$COMMIT_SHA
      - --region=us-central1

images:
  - "us-central1-docker.pkg.dev/$PROJECT_ID/ml-images/api:$COMMIT_SHA"
'''
with open("cloudbuild.yaml", "w") as f:
    f.write(cloudbuild_yaml)
print(cloudbuild_yaml)

In [ ]:
trigger_setup = '''\
gcloud builds triggers create github \\
  --repo-name=YOUR_REPO \\
  --repo-owner=YOUR_GITHUB_ORG \\
  --branch-pattern="^main$" \\
  --build-config=cloudbuild.yaml
'''
print(trigger_setup)
print("Every push to main now: installs deps -> runs tests -> builds image -> pushes")
print("-> deploys to Cloud Run, exactly mirroring the GitHub Actions jobs from Session 10")
print("but running entirely on GCP-native infrastructure instead.")

## Step 4 — Putting the whole loop together

| Trigger | Action | GCP tool |
|---|---|---|
| New data lands / nightly schedule | Check if retraining is warranted, submit pipeline if so | Cloud Scheduler + Cloud Function |
| Pipeline code changes (git push) | Build, test, push, deploy the serving container | Cloud Build |
| Pipeline run completes | Register new model, gate on quality (Session 12's `evaluate_gate`) | Vertex AI Pipelines |
| Deployed endpoint receives traffic | Monitor for drift, feed back into "should we retrain?" | Vertex AI Model Monitoring / Evidently |

This is the same automation loop conceptually as Session 17's drift-triggered
retraining, expressed with GCP-native scheduling/build tools instead of a
DIY Python script.

## What to try next

* Replace the "500 new rows" threshold with an actual drift score from Session 5's
  Evidently report, so retraining is triggered by *distribution change* rather than
  raw row count.
* Add a rollback step to `cloudbuild.yaml`: if the newly deployed Cloud Run revision's
  error rate spikes, automatically route traffic back to the previous revision.
* Compare this GCP-native automation loop against Session 24's GitHub-Actions-centric
  version of the same idea — both are valid, the right choice depends on which
  ecosystem the rest of your infrastructure already lives in.